# Lab 2 — Four Prompt Designs, Measured
**AI Engineering for Managers · Day 1 · 11:30–12:30**

**Goal:** take one real task and run four different prompt designs against it. Measure tokens, cost and quality for each. The winner is rarely the longest prompt — and now you'll have the evidence.

**The task:** classify incoming service tickets for Axion Controls — product, fault code, severity, and whether a known service bulletin applies. This is exactly the shape of task your teams will automate first.

In [ ]:
# SETUP — run this first. It is the same in every lab.
import os, time, json

from openai import OpenAI, AzureOpenAI

ENDPOINT       = os.environ.get("FOUNDRY_ENDPOINT", "")
API_KEY        = os.environ.get("FOUNDRY_API_KEY", "")
MODEL_MINI     = os.environ.get("MODEL_MINI", "gpt-4o-mini")          # small, cheap workhorse
MODEL_FRONTIER = os.environ.get("MODEL_FRONTIER", "gpt-4o")           # large, expensive
MODEL_EMBED    = os.environ.get("MODEL_EMBED", "text-embedding-3-small")

assert ENDPOINT and API_KEY, (
    "Model endpoint not configured. On the lab VMs these are pre-set — "
    "if you see this, call the trainer."
)

# Works with both Azure OpenAI endpoints and OpenAI-compatible serverless endpoints
if any(h in ENDPOINT for h in ("openai.azure.com", "cognitiveservices", "services.ai.azure.com")):
    client = AzureOpenAI(azure_endpoint=ENDPOINT, api_key=API_KEY, api_version="2024-10-21")
else:
    client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

print("Client ready.")
print("Mini model     :", MODEL_MINI)
print("Frontier model :", MODEL_FRONTIER)

In [ ]:
# Helpers used all day. with_retry keeps the room moving if the endpoint rate-limits briefly.
def with_retry(fn, tries=3, wait=5):
    for a in range(tries):
        try:
            return fn()
        except Exception as e:
            if a == tries - 1: raise
            print(f"  (retrying in {wait*(a+1)}s: {e})")
            time.sleep(wait * (a + 1))

def ask(model, messages, temperature=0.2, **kw):
    """Call a chat model and return text + latency + token counts."""
    t0 = time.perf_counter()
    r = with_retry(lambda: client.chat.completions.create(model=model, messages=messages,
                                                          temperature=temperature, **kw))
    latency = time.perf_counter() - t0
    u = r.usage
    return {
        "model": model,
        "text": r.choices[0].message.content,
        "latency_s": round(latency, 2),
        "input_tokens": u.prompt_tokens,
        "output_tokens": u.completion_tokens,
    }

# Prices in USD per 1 MILLION tokens.
# >>> EDIT these to match the actual prices of your deployments. <<<
PRICE = {
    MODEL_MINI:     {"in": 0.15, "out": 0.60},
    MODEL_FRONTIER: {"in": 2.50, "out": 10.00},
}

def cost_usd(result):
    p = PRICE[result["model"]]
    return round((result["input_tokens"] * p["in"] +
                  result["output_tokens"] * p["out"]) / 1_000_000, 6)

## 2.1 — The test tickets
Five raw ticket texts, straight from the field. Note ticket 3 — it matches a known service bulletin (SB-2025-04: F-07 trips on firmware < 2.4.1 at ≥ 8 kHz switching).

In [ ]:
TICKETS = [
    "Panel keeps showing #### on half the tags since the PLC guys changed the program yesterday.",
    "All our remote IO dropped at the same time this morning, drives all stopped on watchdog. "
    "The HMI was still fine. PLC showed E-22. Firmware 3.1.",
    "VectorDrive on the packing hall trips F-07 overtemp every 45 min. Filters clean. "
    "Running 10kHz for noise. FW 2.4.0.",
    "New AXIO station won't join the bus, BF light flashing. Probably my fault, first time configuring one.",
    "Drive trips F-01 the moment the loaded conveyor restarts after an e-stop. Accel is 3 seconds.",
]

# What a correct classification looks like (our ground truth for scoring):
TRUTH = [
    {"product": "OptiPanel 7",     "fault_code": "-",    "bulletin": None},
    {"product": "AX-900",          "fault_code": "E-22", "bulletin": "SB-2026-01"},
    {"product": "VectorDrive 450", "fault_code": "F-07", "bulletin": "SB-2025-04"},
    {"product": "AXIO-16",         "fault_code": "BF",   "bulletin": None},
    {"product": "VectorDrive 450", "fault_code": "F-01", "bulletin": None},
]
print(f"{len(TICKETS)} test tickets loaded.")

## 2.2 — The four designs

| # | Design | The bet it makes |
|---|--------|------------------|
| V1 | **Naive** — one line | "The model will figure it out" |
| V2 | **Kitchen sink** — long prose instructions | "More words = more control" |
| V3 | **Few-shot** — 2 worked examples | "Show, don't tell" |
| V4 | **Structured** — short rules + strict JSON schema + reference data | "Constrain the output, supply the facts" |

In [ ]:
V1_SYSTEM = "Classify this support ticket."

V2_SYSTEM = """You are an extremely capable and helpful senior support ticket classification specialist
working for Axion Controls, a leading manufacturer of industrial automation equipment including
programmable logic controllers, variable frequency drives, remote IO modules and HMI panels.
Your job, which is very important to the business, is to carefully read each incoming support
ticket, think about what product it concerns, determine the fault code if one is present or
implied, assess how severe the issue appears to be, and also consider whether any known service
bulletins might apply to the situation described. Known bulletins: SB-2025-04 concerns
VectorDrive 450 drives on firmware below 2.4.1 tripping F-07 at switching frequencies of 8 kHz
or above; SB-2026-01 concerns AX-900 controllers on firmware 3.2 or below with E-22 ETH2 port
lockups. Please provide your classification in a clear format with the product name, fault code,
severity assessment, applicable bulletin if any, and a brief explanation of your reasoning,
making sure to be thorough and accurate."""

V3_SYSTEM = """Classify Axion Controls support tickets. Products: AX-900 (PLC), VectorDrive 450
(drive), AXIO-16 (remote I/O), OptiPanel 7 (HMI).
Bulletins: SB-2025-04 = VectorDrive F-07, fw<2.4.1, >=8kHz. SB-2026-01 = AX-900 E-22, fw<=3.2.

Example 1:
Ticket: "drive trips F-11 after weekend washdown"
Answer: {"product": "VectorDrive 450", "fault_code": "F-11", "severity": "Medium", "bulletin": null}

Example 2:
Ticket: "controller logs E-22, all profinet gone, fw 3.0"
Answer: {"product": "AX-900", "fault_code": "E-22", "severity": "High", "bulletin": "SB-2026-01"}

Reply with JSON only."""

V4_SYSTEM = """Classify the ticket. JSON only, schema:
{"product": "AX-900|VectorDrive 450|AXIO-16|OptiPanel 7",
 "fault_code": "<code or '-'>", "severity": "Low|Medium|High",
 "bulletin": "SB-2025-04|SB-2026-01|null"}
Bulletin rules: SB-2025-04 iff VectorDrive F-07 AND fw<2.4.1 AND switching>=8kHz.
SB-2026-01 iff AX-900 E-22 AND fw<=3.2."""

DESIGNS = {"V1 naive": V1_SYSTEM, "V2 kitchen sink": V2_SYSTEM,
           "V3 few-shot": V3_SYSTEM, "V4 structured": V4_SYSTEM}
print("Four designs defined.")

## 2.3 — Run the bake-off
20 calls (4 designs × 5 tickets), all on the mini model. We score each answer on three fields: product, fault code, bulletin.

In [ ]:
import re, pandas as pd

def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m: return {}
    try: return json.loads(m.group(0))
    except json.JSONDecodeError: return {}

def score(parsed, truth):
    pts = 0
    if parsed.get("product") == truth["product"]: pts += 1
    if str(parsed.get("fault_code", "")).upper() == truth["fault_code"].upper(): pts += 1
    b = parsed.get("bulletin");  b = None if b in (None, "null", "None", "") else b
    if b == truth["bulletin"]: pts += 1
    return pts  # out of 3

results = []
for name, system in DESIGNS.items():
    tok_in = tok_out = usd = points = 0
    for ticket, truth in zip(TICKETS, TRUTH):
        r = ask(MODEL_MINI, [{"role": "system", "content": system},
                             {"role": "user", "content": ticket}])
        tok_in += r["input_tokens"]; tok_out += r["output_tokens"]; usd += cost_usd(r)
        points += score(extract_json(r["text"]), truth)
    results.append({"design": name, "quality_pct": round(100 * points / (3 * len(TICKETS))),
                    "input_tokens": tok_in, "output_tokens": tok_out,
                    "cost_5_tickets_usd": round(usd, 5),
                    "cost_per_100k_tickets_usd": round(usd / 5 * 100_000, 0)})

df = pd.DataFrame(results)
df

## 2.4 — Read the table like a manager

- **Quality:** which designs actually caught the SB-2025-04 case on ticket 3? That's the one that saves a pointless site visit.
- **Cost:** V2 probably costs several times V4 *per ticket* for equal or worse quality. At 100k tickets/year that column is a real budget line.
- **The cache column you can't see yet:** V4's short, *stable* system prompt is also cache-friendly — providers charge cached input tokens at a fraction of the price. Long prompts that change every call never cache. (More in M6 tomorrow.)

✏️ **YOUR TURN:**
1. Change one thing in V4 (an instruction, the schema, a rule) and re-run 2.3. Did quality move? Did tokens?
2. Write the first entry of your team's **prompt library** below — the pattern you'd standardise.

In [ ]:
PROMPT_LIBRARY_ENTRY = """
Task: ticket classification
Standard design: (which V, and why)
Owner:
Measured: quality ___% at $___ per 100k tickets, model: mini
"""
print(PROMPT_LIBRARY_ENTRY)

## 2.5 — Stretch
Run V4 on the frontier model and compare with V4-on-mini. Is the frontier model worth 15–20× the price *for this task*? (This is the model-routing decision in M7 tomorrow.)

## 2.6 — Result line
Copy the printed line into the **Paste & check** box for Lab 2 on the course page.

In [ ]:
best = max(results, key=lambda r: (r["quality_pct"], -r["cost_5_tickets_usd"]))
print(f'LAB2|best={best["design"].split()[0]}|quality={best["quality_pct"]}|cost_100k={best["cost_per_100k_tickets_usd"]:.0f}')

✅ **Done.** You have measured proof that prompt design is an engineering decision with a budget attached — and the start of a prompt library.